In [1]:
import sys
sys.path.append("..")

from src.logging_config import setup_logging
setup_logging()

import logging
from src.data import inspect_h5, load_split, DATA_PATH

logger = logging.getLogger("phase0")

Locate the dataset file

In [2]:
h5_files = list(DATA_PATH.glob("*.h5"))
H5 = h5_files[0]
logger.info("Using: %s", H5.name)

21:44:37 | INFO    | phase0 | Using: N-CMAPSS_DS02-006.h5


HDF5 schema 

In [3]:
inspect_h5(H5)


21:44:57 | INFO    | src.data | File: /Users/alan/Desktop/ML_Final_Project/data/raw/N-CMAPSS_DS02-006.h5
21:44:57 | INFO    | src.data | Top-level keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
21:44:57 | INFO    | src.data | All datasets:
21:44:57 | INFO    | src.data |   A_dev                shape=(5263447, 4)         dtype=float64
21:44:57 | INFO    | src.data |   A_test               shape=(1253743, 4)         dtype=float64
21:44:57 | INFO    | src.data |   A_var                shape=(4,)                 dtype=|S5
21:44:57 | INFO    | src.data |   T_dev                shape=(5263447, 10)        dtype=float64
21:44:57 | INFO    | src.data |   T_test               shape=(1253743, 10)        dtype=float64
21:44:57 | INFO    | src.data |   T_var                shape=(10,)                dtype=|S12
21:44:57 | INFO    | src.data |   W_dev                shape

Load both splits

In [4]:
dev  = load_split(H5, split="dev")
test = load_split(H5, split="test")

logger.info("dev shape:  %s", dev.shape)
logger.info("test shape: %s", test.shape)
logger.info("columns: %s", list(dev.columns))

21:45:24 | INFO    | src.data | Loading split=dev from N-CMAPSS_DS02-006.h5
21:45:25 | INFO    | src.data | Loaded split=dev: shape=(5263447, 23), memory=0.48 GB
21:45:25 | INFO    | src.data | Loading split=test from N-CMAPSS_DS02-006.h5
21:45:25 | INFO    | src.data | Loaded split=test: shape=(1253743, 23), memory=0.12 GB
21:45:25 | INFO    | phase0 | dev shape:  (5263447, 23)
21:45:25 | INFO    | phase0 | test shape: (1253743, 23)
21:45:25 | INFO    | phase0 | columns: ['alt', 'Mach', 'TRA', 'T2', 'T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf', 'unit', 'cycle', 'Fc', 'hs', 'RUL']


Per-unit summaries

In [5]:
dev_summary = (
    dev.groupby("unit")
       .agg(samples=("RUL", "size"),
            max_cycle=("cycle", "max"),
            min_RUL=("RUL", "min"),
            max_RUL=("RUL", "max"))
       .round(2)
)
logger.info("Dev units:\n%s", dev_summary)

test_summary = (
    test.groupby("unit")
        .agg(samples=("RUL", "size"),
             max_cycle=("cycle", "max"),
             min_RUL=("RUL", "min"),
             max_RUL=("RUL", "max"))
        .round(2)
)
logger.info("Test units:\n%s", test_summary)

21:46:01 | INFO    | phase0 | Dev units:
      samples  max_cycle  min_RUL  max_RUL
unit                                      
2      853142         75      0.0     74.0
5     1033420         89      0.0     88.0
10     952711         82      0.0     81.0
16     765295         63      0.0     62.0
18     890719         71      0.0     70.0
20     768160         66      0.0     65.0
21:46:01 | INFO    | phase0 | Test units:
      samples  max_cycle  min_RUL  max_RUL
unit                                      
11     663495         59      0.0     58.0
14     156778         76      0.0     75.0
15     433470         67      0.0     66.0


Descriptive stats on flight conditions and a few key sensors

In [6]:
flight_cols = ["alt", "Mach", "TRA", "T2"]
logger.info("Flight conditions (dev):\n%s", dev[flight_cols].describe().round(2))

sensor_cols = ["Wf", "Nf", "Nc", "T48", "Ps30"]
logger.info("Selected sensors (dev):\n%s", dev[sensor_cols].describe().round(2))

21:46:36 | INFO    | phase0 | Flight conditions (dev):
              alt        Mach         TRA          T2
count  5263447.00  5263447.00  5263447.00  5263447.00
mean     21901.03        0.62       68.55      475.01
std       6288.50        0.08       14.55       16.59
min      10001.00        0.26       23.55      421.38
25%      16451.00        0.57       59.41      460.88
50%      22996.00        0.64       74.62      473.04
75%      27912.00        0.69       79.63      489.37
max      35033.00        0.75       87.63      510.81
21:46:36 | INFO    | phase0 | Selected sensors (dev):
               Wf          Nf          Nc         T48        Ps30
count  5263447.00  5263447.00  5263447.00  5263447.00  5263447.00
mean         2.25     2014.10     8204.63     1642.64      208.18
std          0.57      138.68      184.07       98.53       43.75
min          0.33     1469.74     7366.11      944.50       80.33
25%          1.87     1959.93     8106.51     1594.24      177.73
50%      

RUL distribution

In [7]:
logger.info("RUL distribution (dev):\n%s", dev["RUL"].describe().round(2))
logger.info("RUL distribution (test):\n%s", test["RUL"].describe().round(2))

21:47:04 | INFO    | phase0 | RUL distribution (dev):
count    5263447.00
mean          37.33
std           22.37
min            0.00
25%           18.00
50%           36.00
75%           56.00
max           88.00
Name: RUL, dtype: float64
21:47:04 | INFO    | phase0 | RUL distribution (test):
count    1253743.00
mean          31.29
std           18.97
min            0.00
25%           15.00
50%           31.00
75%           48.00
max           75.00
Name: RUL, dtype: float64
